# MLP on Adiac dataset

In [5]:
# %load MLP.py
#!/usr/bin/env python2
"""
Created on Fri Oct 28 21:46:23 2016

@author: stephen
"""
 
from __future__ import print_function
 
from tensorflow import keras
import numpy as np
import pandas as pd

np.random.seed(813306)

      
def readucr(filename):
    data = np.loadtxt(filename, delimiter = ',')
    Y = data[:,0]
    X = data[:,1:]
    return X, Y
  
nb_epochs = 5000

     
#flist = ['Adiac', 'Beef', 'CBF', 'ChlorineConcentration', 'CinC_ECG_torso', 'Coffee', 'Cricket_X', 'Cricket_Y', 'Cricket_Z', 
#'DiatomSizeReduction', 'ECGFiveDays', 'FaceAll', 'FaceFour', 'FacesUCR', '50words', 'FISH', 'Gun_Point', 'Haptics', 
#'InlineSkate', 'ItalyPowerDemand', 'Lighting2', 'Lighting7', 'MALLAT', 'MedicalImages', 'MoteStrain', 'NonInvasiveFatalECG_Thorax1', 
#'NonInvasiveFatalECG_Thorax2', 'OliveOil', 'OSULeaf', 'SonyAIBORobotSurface', 'SonyAIBORobotSurfaceII', 'StarLightCurves', 'SwedishLeaf', 'Symbols', 
#'synthetic_control', 'Trace', 'TwoLeadECG', 'Two_Patterns', 'uWaveGestureLibrary_X', 'uWaveGestureLibrary_Y', 'uWaveGestureLibrary_Z', 'wafer', 'WordsSynonyms', 'yoga']

flist = ['Adiac']
for each in flist:
    fname = each
    x_train, y_train = readucr(fname+'/'+fname+'_TRAIN')
    x_test, y_test = readucr(fname+'/'+fname+'_TEST')
    nb_classes =len(np.unique(y_test))
    y_train = (y_train - y_train.min())/(y_train.max()-y_train.min())*(nb_classes-1)
    y_test = (y_test - y_test.min())/(y_test.max()-y_test.min())*(nb_classes-1)
    batch_size = min(x_train.shape[0]/10, 16)
    
    Y_train = keras.utils.to_categorical(y_train, nb_classes)
    Y_test = keras.utils.to_categorical(y_test, nb_classes)
     
    x_train_mean = x_train.mean()
    x_train_std = x_train.std()
    x_train = (x_train - x_train_mean)/(x_train_std)
     
   # x_test_min = np.min(x_test, axis = 1, keepdims=1)
   # x_test_max = np.max(x_test, axis = 1, keepdims=1)
    x_test = (x_test - x_train_mean)/(x_train_std)
     
    #x_train = x_train.reshape(x_train.shape + (1,))
    #x_test = x_test.reshape(x_test.shape + (1,))
    
    x = keras.layers.Input(x_train.shape[1:])
    y= keras.layers.Dropout(0.1)(x)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation = 'relu')(y)
    y = keras.layers.Dropout(0.3)(y)
    out = keras.layers.Dense(nb_classes, activation='softmax')(y)
     
    model = keras.models.Model(inputs=x, outputs=out)
     
    optimizer = keras.optimizers.Adadelta()    
    model.compile(loss='categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
     
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor = 'loss', factor=0.5,
                      patience=200, min_lr=0.1)
    
    hist = model.fit(x_train, Y_train, batch_size=batch_size, epochs=nb_epochs,
              verbose=1, validation_data=(x_test, Y_test), 
                #callbacks = [TestCallback((x_train, Y_train)), reduce_lr, keras.callbacks.TensorBoard(log_dir='./log'+fname, histogram_freq=1)])
                 callbacks=[reduce_lr])
    
    #Print the testing results which has the lowest training loss.
    log = pd.DataFrame(hist.history)
    print(log.loc[log['loss'].idxmin()]['loss'], log.loc[log['loss'].idxmin()]['val_accuracy'])

Epoch 1/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.0282 - loss: 3.7544 - val_accuracy: 0.0281 - val_loss: 3.6545 - learning_rate: 0.0010
Epoch 2/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0179 - loss: 3.7489 - val_accuracy: 0.0281 - val_loss: 3.6531 - learning_rate: 0.0010
Epoch 3/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0333 - loss: 3.7286 - val_accuracy: 0.0281 - val_loss: 3.6519 - learning_rate: 0.0010
Epoch 4/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0128 - loss: 3.7299 - val_accuracy: 0.0281 - val_loss: 3.6508 - learning_rate: 0.0010
Epoch 5/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0282 - loss: 3.7329 - val_accuracy: 0.0281 - val_loss: 3.6496 - learning_rate: 0.0010
Epoch 6/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0231 - loss: 3.7297 - val_accuracy: 0.0281 - val_loss: 3.6484 - learning_rate: 0.0010
Epoch 7/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0410 - loss: 3.7110 - 

# Debugged MLP on Adiac dataset

In [1]:
# %load MLP.py
#!/usr/bin/env python2
"""
Created on Fri Oct 28 21:46:23 2016

@author: stephen
"""
 
from __future__ import print_function
 
from tensorflow import keras
import numpy as np
import pandas as pd

np.random.seed(813306)

      
def readucr(filename):
    data = np.loadtxt(filename, delimiter = ',')
    Y = data[:,0]
    X = data[:,1:]
    return X, Y
  
nb_epochs = 5000

     
#flist = ['Adiac', 'Beef', 'CBF', 'ChlorineConcentration', 'CinC_ECG_torso', 'Coffee', 'Cricket_X', 'Cricket_Y', 'Cricket_Z', 
#'DiatomSizeReduction', 'ECGFiveDays', 'FaceAll', 'FaceFour', 'FacesUCR', '50words', 'FISH', 'Gun_Point', 'Haptics', 
#'InlineSkate', 'ItalyPowerDemand', 'Lighting2', 'Lighting7', 'MALLAT', 'MedicalImages', 'MoteStrain', 'NonInvasiveFatalECG_Thorax1', 
#'NonInvasiveFatalECG_Thorax2', 'OliveOil', 'OSULeaf', 'SonyAIBORobotSurface', 'SonyAIBORobotSurfaceII', 'StarLightCurves', 'SwedishLeaf', 'Symbols', 
#'synthetic_control', 'Trace', 'TwoLeadECG', 'Two_Patterns', 'uWaveGestureLibrary_X', 'uWaveGestureLibrary_Y', 'uWaveGestureLibrary_Z', 'wafer', 'WordsSynonyms', 'yoga']

flist = ['Adiac']
for each in flist:
    fname = each
    x_train, y_train = readucr(fname+'/'+fname+'_TRAIN')
    x_test, y_test = readucr(fname+'/'+fname+'_TEST')
    nb_classes =len(np.unique(y_test))
    y_train = (y_train - y_train.min())/(y_train.max()-y_train.min())*(nb_classes-1)
    y_test = (y_test - y_test.min())/(y_test.max()-y_test.min())*(nb_classes-1)
    batch_size = min(x_train.shape[0]/10, 16)
    
    Y_train = keras.utils.to_categorical(y_train, nb_classes)
    Y_test = keras.utils.to_categorical(y_test, nb_classes)
     
    x_train_mean = x_train.mean()
    x_train_std = x_train.std()
    x_train = (x_train - x_train_mean)/(x_train_std)
     
   # x_test_min = np.min(x_test, axis = 1, keepdims=1)
   # x_test_max = np.max(x_test, axis = 1, keepdims=1)
    x_test = (x_test - x_train_mean)/(x_train_std)
     
    #x_train = x_train.reshape(x_train.shape + (1,))
    #x_test = x_test.reshape(x_test.shape + (1,))
    
    x = keras.layers.Input(x_train.shape[1:])
    y= keras.layers.Dropout(0.1)(x)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation = 'relu')(y)
    y = keras.layers.Dropout(0.3)(y)
    out = keras.layers.Dense(nb_classes, activation='softmax')(y)
     
    model = keras.models.Model(inputs=x, outputs=out)
     
    optimizer = keras.optimizers.Adadelta(learning_rate = 1.0)    
    model.compile(loss='categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
     
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor = 'loss', factor=0.5,
                      patience=200, min_lr=0.1)
    
    hist = model.fit(x_train, Y_train, batch_size=batch_size, epochs=nb_epochs,
              verbose=1, validation_data=(x_test, Y_test), 
                #callbacks = [TestCallback((x_train, Y_train)), reduce_lr, keras.callbacks.TensorBoard(log_dir='./log'+fname, histogram_freq=1)])
                 callbacks=[reduce_lr])
    
    #Print the testing results which has the lowest training loss.
    log = pd.DataFrame(hist.history)
    print(log.loc[log['loss'].idxmin()]['loss'], log.loc[log['loss'].idxmin()]['val_accuracy'])

Epoch 1/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.0308 - loss: 3.7126 - val_accuracy: 0.0742 - val_loss: 3.5559 - learning_rate: 1.0000
Epoch 2/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0667 - loss: 3.5431 - val_accuracy: 0.0844 - val_loss: 3.4644 - learning_rate: 1.0000
Epoch 3/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0692 - loss: 3.4384 - val_accuracy: 0.1023 - val_loss: 3.3636 - learning_rate: 1.0000
Epoch 4/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1051 - loss: 3.3615 - val_accuracy: 0.1049 - val_loss: 3.3348 - learning_rate: 1.0000
Epoch 5/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1077 - loss: 3.3110 - val_accuracy: 0.0895 - val_loss: 3.3293 - learning_rate: 1.0000
Epoch 6/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1051 - loss: 3.2912 - val_accuracy: 0.1074 - val_loss: 3.2570 - learning_rate: 1.0000
Epoch 7/5000
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1051 - loss: 3.2281 - 

# MLP on GunPoint dataset

In [1]:
# %load MLP.py
#!/usr/bin/env python2
"""
Created on Fri Oct 28 21:46:23 2016

@author: stephen
"""
 
from __future__ import print_function
 
from tensorflow import keras
import numpy as np
import pandas as pd

np.random.seed(813306)

      
def readucr(filename):
    data = np.loadtxt(filename + '.txt')
    Y = data[:,0]
    X = data[:,1:]
    return X, Y
  
nb_epochs = 5000

     
#flist = ['Adiac', 'Beef', 'CBF', 'ChlorineConcentration', 'CinC_ECG_torso', 'Coffee', 'Cricket_X', 'Cricket_Y', 'Cricket_Z', 
#'DiatomSizeReduction', 'ECGFiveDays', 'FaceAll', 'FaceFour', 'FacesUCR', '50words', 'FISH', 'Gun_Point', 'Haptics', 
#'InlineSkate', 'ItalyPowerDemand', 'Lighting2', 'Lighting7', 'MALLAT', 'MedicalImages', 'MoteStrain', 'NonInvasiveFatalECG_Thorax1', 
#'NonInvasiveFatalECG_Thorax2', 'OliveOil', 'OSULeaf', 'SonyAIBORobotSurface', 'SonyAIBORobotSurfaceII', 'StarLightCurves', 'SwedishLeaf', 'Symbols', 
#'synthetic_control', 'Trace', 'TwoLeadECG', 'Two_Patterns', 'uWaveGestureLibrary_X', 'uWaveGestureLibrary_Y', 'uWaveGestureLibrary_Z', 'wafer', 'WordsSynonyms', 'yoga']

flist = ['GunPoint']
for each in flist:
    fname = each
    x_train, y_train = readucr(fname+'/'+fname+'_TRAIN')
    x_test, y_test = readucr(fname+'/'+fname+'_TEST')
    nb_classes =len(np.unique(y_test))
    y_train = (y_train - y_train.min())/(y_train.max()-y_train.min())*(nb_classes-1)
    y_test = (y_test - y_test.min())/(y_test.max()-y_test.min())*(nb_classes-1)
    batch_size = min(x_train.shape[0] // 10, 16)
    
    Y_train = keras.utils.to_categorical(y_train, nb_classes)
    Y_test = keras.utils.to_categorical(y_test, nb_classes)
     
    x_train_mean = x_train.mean()
    x_train_std = x_train.std()
    x_train = (x_train - x_train_mean)/(x_train_std)
     
   # x_test_min = np.min(x_test, axis = 1, keepdims=1)
   # x_test_max = np.max(x_test, axis = 1, keepdims=1)
    x_test = (x_test - x_train_mean)/(x_train_std)
     
    #x_train = x_train.reshape(x_train.shape + (1,))
    #x_test = x_test.reshape(x_test.shape + (1,))
    
    x = keras.layers.Input(x_train.shape[1:])
    y= keras.layers.Dropout(0.1)(x)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation = 'relu')(y)
    y = keras.layers.Dropout(0.3)(y)
    out = keras.layers.Dense(nb_classes, activation='softmax')(y)
     
    model = keras.models.Model(inputs=x, outputs=out)
     
    optimizer = keras.optimizers.Adadelta(learning_rate = 1.0)    
    model.compile(loss='categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
     
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor = 'loss', factor=0.5,
                      patience=200, min_lr=0.1)
    
    hist = model.fit(x_train, Y_train, batch_size=batch_size, epochs=nb_epochs,
              verbose=1, validation_data=(x_test, Y_test), 
                #callbacks = [TestCallback((x_train, Y_train)), reduce_lr, keras.callbacks.TensorBoard(log_dir='./log'+fname, histogram_freq=1)])
                 callbacks=[reduce_lr])
    
    #Print the testing results which has the lowest training loss.
    log = pd.DataFrame(hist.history)
    print(log.loc[log['loss'].idxmin()]['loss'], log.loc[log['loss'].idxmin()]['val_accuracy'])

Epoch 1/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.2800 - loss: 1.0864 - val_accuracy: 0.5800 - val_loss: 0.7126 - learning_rate: 1.0000
Epoch 2/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6600 - loss: 0.6085 - val_accuracy: 0.6800 - val_loss: 0.5196 - learning_rate: 1.0000
Epoch 3/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8200 - loss: 0.3817 - val_accuracy: 0.6800 - val_loss: 0.6600 - learning_rate: 1.0000
Epoch 4/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7800 - loss: 0.4207 - val_accuracy: 0.7800 - val_loss: 0.4063 - learning_rate: 1.0000
Epoch 5/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8200 - loss: 0.5184 - val_accuracy: 0.7667 - val_loss: 0.4042 - learning_rate: 1.0000
Epoch 6/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8400 - loss: 0.3111 - val_accuracy: 0.7067 - val_loss: 0.6347 - learning_rate: 1.0000
Epoch 7/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9000 - loss: 0.2393 -

# MLP on InlineSkate dataset

In [2]:
# %load MLP.py
#!/usr/bin/env python2
"""
Created on Fri Oct 28 21:46:23 2016

@author: stephen
"""
 
from __future__ import print_function
 
from tensorflow import keras
import numpy as np
import pandas as pd

np.random.seed(813306)

      
def readucr(filename):
    data = np.loadtxt(filename + '.txt')
    Y = data[:,0]
    X = data[:,1:]
    return X, Y
  
nb_epochs = 5000

     
#flist = ['Adiac', 'Beef', 'CBF', 'ChlorineConcentration', 'CinC_ECG_torso', 'Coffee', 'Cricket_X', 'Cricket_Y', 'Cricket_Z', 
#'DiatomSizeReduction', 'ECGFiveDays', 'FaceAll', 'FaceFour', 'FacesUCR', '50words', 'FISH', 'Gun_Point', 'Haptics', 
#'InlineSkate', 'ItalyPowerDemand', 'Lighting2', 'Lighting7', 'MALLAT', 'MedicalImages', 'MoteStrain', 'NonInvasiveFatalECG_Thorax1', 
#'NonInvasiveFatalECG_Thorax2', 'OliveOil', 'OSULeaf', 'SonyAIBORobotSurface', 'SonyAIBORobotSurfaceII', 'StarLightCurves', 'SwedishLeaf', 'Symbols', 
#'synthetic_control', 'Trace', 'TwoLeadECG', 'Two_Patterns', 'uWaveGestureLibrary_X', 'uWaveGestureLibrary_Y', 'uWaveGestureLibrary_Z', 'wafer', 'WordsSynonyms', 'yoga']

flist = ['InlineSkate']
for each in flist:
    fname = each
    x_train, y_train = readucr(fname+'/'+fname+'_TRAIN')
    x_test, y_test = readucr(fname+'/'+fname+'_TEST')
    nb_classes =len(np.unique(y_test))
    y_train = (y_train - y_train.min())/(y_train.max()-y_train.min())*(nb_classes-1)
    y_test = (y_test - y_test.min())/(y_test.max()-y_test.min())*(nb_classes-1)
    batch_size = min(x_train.shape[0] // 10, 16)
    
    Y_train = keras.utils.to_categorical(y_train, nb_classes)
    Y_test = keras.utils.to_categorical(y_test, nb_classes)
     
    x_train_mean = x_train.mean()
    x_train_std = x_train.std()
    x_train = (x_train - x_train_mean)/(x_train_std)
     
   # x_test_min = np.min(x_test, axis = 1, keepdims=1)
   # x_test_max = np.max(x_test, axis = 1, keepdims=1)
    x_test = (x_test - x_train_mean)/(x_train_std)
     
    #x_train = x_train.reshape(x_train.shape + (1,))
    #x_test = x_test.reshape(x_test.shape + (1,))
    
    x = keras.layers.Input(x_train.shape[1:])
    y= keras.layers.Dropout(0.1)(x)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation = 'relu')(y)
    y = keras.layers.Dropout(0.3)(y)
    out = keras.layers.Dense(nb_classes, activation='softmax')(y)
     
    model = keras.models.Model(inputs=x, outputs=out)
     
    optimizer = keras.optimizers.Adadelta(learning_rate = 1.0)    
    model.compile(loss='categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
     
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor = 'loss', factor=0.5,
                      patience=200, min_lr=0.1)
    
    hist = model.fit(x_train, Y_train, batch_size=batch_size, epochs=nb_epochs,
              verbose=1, validation_data=(x_test, Y_test), 
                #callbacks = [TestCallback((x_train, Y_train)), reduce_lr, keras.callbacks.TensorBoard(log_dir='./log'+fname, histogram_freq=1)])
                 callbacks=[reduce_lr])
    
    #Print the testing results which has the lowest training loss.
    log = pd.DataFrame(hist.history)
    print(log.loc[log['loss'].idxmin()]['loss'], log.loc[log['loss'].idxmin()]['val_accuracy'])

Epoch 1/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1500 - loss: 3.0211 - val_accuracy: 0.1582 - val_loss: 2.6636 - learning_rate: 1.0000
Epoch 2/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2100 - loss: 2.4157 - val_accuracy: 0.1982 - val_loss: 2.0921 - learning_rate: 1.0000
Epoch 3/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2200 - loss: 2.0468 - val_accuracy: 0.1709 - val_loss: 2.2449 - learning_rate: 1.0000
Epoch 4/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2400 - loss: 1.9955 - val_accuracy: 0.1964 - val_loss: 2.0577 - learning_rate: 1.0000
Epoch 5/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3800 - loss: 1.7030 - val_accuracy: 0.2182 - val_loss: 2.1215 - learning_rate: 1.0000
Epoch 6/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3600 - loss: 1.6999 - val_accuracy: 0.2418 - val_loss: 2.1716 - learning_rate: 1.0000
Epoch 7/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2200 - loss: 1.

# MLP on MedicalImages dataset

In [3]:
# %load MLP.py
#!/usr/bin/env python2
"""
Created on Fri Oct 28 21:46:23 2016

@author: stephen
"""
 
from __future__ import print_function
 
from tensorflow import keras
import numpy as np
import pandas as pd

np.random.seed(813306)

      
def readucr(filename):
    data = np.loadtxt(filename + '.txt')
    Y = data[:,0]
    X = data[:,1:]
    return X, Y
  
nb_epochs = 5000

     
#flist = ['Adiac', 'Beef', 'CBF', 'ChlorineConcentration', 'CinC_ECG_torso', 'Coffee', 'Cricket_X', 'Cricket_Y', 'Cricket_Z', 
#'DiatomSizeReduction', 'ECGFiveDays', 'FaceAll', 'FaceFour', 'FacesUCR', '50words', 'FISH', 'Gun_Point', 'Haptics', 
#'InlineSkate', 'ItalyPowerDemand', 'Lighting2', 'Lighting7', 'MALLAT', 'MedicalImages', 'MoteStrain', 'NonInvasiveFatalECG_Thorax1', 
#'NonInvasiveFatalECG_Thorax2', 'OliveOil', 'OSULeaf', 'SonyAIBORobotSurface', 'SonyAIBORobotSurfaceII', 'StarLightCurves', 'SwedishLeaf', 'Symbols', 
#'synthetic_control', 'Trace', 'TwoLeadECG', 'Two_Patterns', 'uWaveGestureLibrary_X', 'uWaveGestureLibrary_Y', 'uWaveGestureLibrary_Z', 'wafer', 'WordsSynonyms', 'yoga']

flist = ['MedicalImages']
for each in flist:
    fname = each
    x_train, y_train = readucr(fname+'/'+fname+'_TRAIN')
    x_test, y_test = readucr(fname+'/'+fname+'_TEST')
    nb_classes =len(np.unique(y_test))
    y_train = (y_train - y_train.min())/(y_train.max()-y_train.min())*(nb_classes-1)
    y_test = (y_test - y_test.min())/(y_test.max()-y_test.min())*(nb_classes-1)
    batch_size = min(x_train.shape[0] // 10, 16)
    
    Y_train = keras.utils.to_categorical(y_train, nb_classes)
    Y_test = keras.utils.to_categorical(y_test, nb_classes)
     
    x_train_mean = x_train.mean()
    x_train_std = x_train.std()
    x_train = (x_train - x_train_mean)/(x_train_std)
     
   # x_test_min = np.min(x_test, axis = 1, keepdims=1)
   # x_test_max = np.max(x_test, axis = 1, keepdims=1)
    x_test = (x_test - x_train_mean)/(x_train_std)
     
    #x_train = x_train.reshape(x_train.shape + (1,))
    #x_test = x_test.reshape(x_test.shape + (1,))
    
    x = keras.layers.Input(x_train.shape[1:])
    y= keras.layers.Dropout(0.1)(x)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation = 'relu')(y)
    y = keras.layers.Dropout(0.3)(y)
    out = keras.layers.Dense(nb_classes, activation='softmax')(y)
     
    model = keras.models.Model(inputs=x, outputs=out)
     
    optimizer = keras.optimizers.Adadelta(learning_rate = 1.0)    
    model.compile(loss='categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
     
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor = 'loss', factor=0.5,
                      patience=200, min_lr=0.1)
    
    hist = model.fit(x_train, Y_train, batch_size=batch_size, epochs=nb_epochs,
              verbose=1, validation_data=(x_test, Y_test), 
                #callbacks = [TestCallback((x_train, Y_train)), reduce_lr, keras.callbacks.TensorBoard(log_dir='./log'+fname, histogram_freq=1)])
                 callbacks=[reduce_lr])
    
    #Print the testing results which has the lowest training loss.
    log = pd.DataFrame(hist.history)
    print(log.loc[log['loss'].idxmin()]['loss'], log.loc[log['loss'].idxmin()]['val_accuracy'])

Epoch 1/5000
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4987 - loss: 1.6457 - val_accuracy: 0.4684 - val_loss: 1.3985 - learning_rate: 1.0000
Epoch 2/5000
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5722 - loss: 1.2098 - val_accuracy: 0.6118 - val_loss: 1.1006 - learning_rate: 1.0000
Epoch 3/5000
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5801 - loss: 1.1059 - val_accuracy: 0.6066 - val_loss: 1.0511 - learning_rate: 1.0000
Epoch 4/5000
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6247 - loss: 1.0092 - val_accuracy: 0.5724 - val_loss: 1.0322 - learning_rate: 1.0000
Epoch 5/5000
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6273 - loss: 0.9262 - val_accuracy: 0.5750 - val_loss: 1.0352 - learning_rate: 1.0000
Epoch 6/5000
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6798 - loss: 0.8336 - val_accuracy: 0.6447 - val_loss: 0.9301 - learning_rate: 1.0000
Epoch 7/5000
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6745 - loss: 0.8209 - 

# MLP on Coffee dataset

In [5]:
# %load MLP.py
#!/usr/bin/env python2
"""
Created on Fri Oct 28 21:46:23 2016

@author: stephen
"""
 
from __future__ import print_function
 
from tensorflow import keras
import numpy as np
import pandas as pd

np.random.seed(813306)

      
def readucr(filename):
    data = np.loadtxt(filename + '.txt')
    Y = data[:,0]
    X = data[:,1:]
    return X, Y
  
nb_epochs = 5000

     
#flist = ['Adiac', 'Beef', 'CBF', 'ChlorineConcentration', 'CinC_ECG_torso', 'Coffee', 'Cricket_X', 'Cricket_Y', 'Cricket_Z', 
#'DiatomSizeReduction', 'ECGFiveDays', 'FaceAll', 'FaceFour', 'FacesUCR', '50words', 'FISH', 'Gun_Point', 'Haptics', 
#'InlineSkate', 'ItalyPowerDemand', 'Lighting2', 'Lighting7', 'MALLAT', 'MedicalImages', 'MoteStrain', 'NonInvasiveFatalECG_Thorax1', 
#'NonInvasiveFatalECG_Thorax2', 'OliveOil', 'OSULeaf', 'SonyAIBORobotSurface', 'SonyAIBORobotSurfaceII', 'StarLightCurves', 'SwedishLeaf', 'Symbols', 
#'synthetic_control', 'Trace', 'TwoLeadECG', 'Two_Patterns', 'uWaveGestureLibrary_X', 'uWaveGestureLibrary_Y', 'uWaveGestureLibrary_Z', 'wafer', 'WordsSynonyms', 'yoga']

flist = ['Coffee']
for each in flist:
    fname = each
    x_train, y_train = readucr(fname+'/'+fname+'_TRAIN')
    x_test, y_test = readucr(fname+'/'+fname+'_TEST')
    nb_classes =len(np.unique(y_test))
    y_train = (y_train - y_train.min())/(y_train.max()-y_train.min())*(nb_classes-1)
    y_test = (y_test - y_test.min())/(y_test.max()-y_test.min())*(nb_classes-1)
    batch_size = min(x_train.shape[0] // 10, 16)
    
    Y_train = keras.utils.to_categorical(y_train, nb_classes)
    Y_test = keras.utils.to_categorical(y_test, nb_classes)
     
    x_train_mean = x_train.mean()
    x_train_std = x_train.std()
    x_train = (x_train - x_train_mean)/(x_train_std)
     
   # x_test_min = np.min(x_test, axis = 1, keepdims=1)
   # x_test_max = np.max(x_test, axis = 1, keepdims=1)
    x_test = (x_test - x_train_mean)/(x_train_std)
     
    #x_train = x_train.reshape(x_train.shape + (1,))
    #x_test = x_test.reshape(x_test.shape + (1,))
    
    x = keras.layers.Input(x_train.shape[1:])
    y= keras.layers.Dropout(0.1)(x)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)
    y = keras.layers.Dense(500, activation = 'relu')(y)
    y = keras.layers.Dropout(0.3)(y)
    out = keras.layers.Dense(nb_classes, activation='softmax')(y)
     
    model = keras.models.Model(inputs=x, outputs=out)
     
    optimizer = keras.optimizers.Adadelta(learning_rate = 1.0)    
    model.compile(loss='categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
     
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor = 'loss', factor=0.5,
                      patience=200, min_lr=0.1)
    
    hist = model.fit(x_train, Y_train, batch_size=batch_size, epochs=nb_epochs,
              verbose=1, validation_data=(x_test, Y_test), 
                #callbacks = [TestCallback((x_train, Y_train)), reduce_lr, keras.callbacks.TensorBoard(log_dir='./log'+fname, histogram_freq=1)])
                 callbacks=[reduce_lr])
    
    #Print the testing results which has the lowest training loss.
    log = pd.DataFrame(hist.history)
    print(log.loc[log['loss'].idxmin()]['loss'], log.loc[log['loss'].idxmin()]['val_accuracy'])

Epoch 1/5000
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6071 - loss: 1.2682 - val_accuracy: 0.5357 - val_loss: 0.7044 - learning_rate: 1.0000
Epoch 2/5000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4286 - loss: 0.9473 - val_accuracy: 0.6071 - val_loss: 0.6646 - learning_rate: 1.0000
Epoch 3/5000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6071 - loss: 0.7322 - val_accuracy: 0.5357 - val_loss: 2.0263 - learning_rate: 1.0000
Epoch 4/5000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4643 - loss: 1.0758 - val_accuracy: 0.5357 - val_loss: 0.6921 - learning_rate: 1.0000
Epoch 5/5000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4286 - loss: 0.7935 - val_accuracy: 0.5357 - val_loss: 0.6598 - learning_rate: 1.0000
Epoch 6/5000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5000 - loss: 0.8467 - val_accuracy: 0.5357 - val_loss: 0.7499 - learning_rate: 1.0000
Epoch 7/5000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5714 - loss: 0.7307 - 